# Demo 13 - Retro-hunt: sweep IOCs across full lake history (HERO)

**Pool:** Medium · **Visual:** first-seen to last-seen activity timeline

**The question:** have we **ever** seen these indicators - not have we seen them recently?

A threat report lands with a list of hashes, addresses and domains. The analytics tier is
cost-capped at around 90 days, so in practice it can only answer the second question. The
data lake keeps up to 12 years cheaply, so this notebook sweeps the entire history in one
pass and reports first-seen and last-seen for every indicator.

This is the data lake's signature move, and the clearest answer to "why does the retention
matter?".

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Parameters - your indicator list

This is the cell you edit. Paste in the hashes, IP addresses and domains from whatever
threat report you are working from.

Mark it as a **Parameters** cell (Cell menu -> Mark Cell as Parameters) and you can run this
whole notebook as a scheduled job with a different IOC list each time.

In [ ]:
# Parameters  (Cell menu -> Mark Cell as Parameters to reuse this as a scheduled retro-hunt)
WORKSPACE = "your-workspace-name"

ioc_sha256  = [
    "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f",  # example EICAR-like
]
ioc_ips     = ["185.220.101.4", "45.137.21.9"]
ioc_domains = ["evil-c2.example.com", "badactor.example.net"]

## 3. Sweep every indicator across the full history

**There is deliberately no time filter in this cell.** That is the entire point of the
notebook.

The analytics tier is cost-capped at roughly 90 days, so in practice "have we ever seen
this?" quietly becomes "have we seen this recently?". The data lake holds up to 12 years
cheaply, so the question can actually be asked properly.

For each indicator we record first seen, last seen, total hits and how many distinct
devices. Tables that do not exist in this workspace, or that lack the column we need, are
skipped with a message rather than crashing the run.

In [ ]:
import pandas as pd

def safe(tbl):
    try:
        return data_provider.read_table(tbl, WORKSPACE)
    except Exception as e:
        print(f"(skip {tbl}: {e})"); return None

hits = []   # rows: type, ioc, table, device, time

# --- File hash IOCs across DeviceFileEvents + DeviceProcessEvents (no time filter) ---
for tbl, hashcol in [("DeviceFileEvents", "SHA256"), ("DeviceProcessEvents", "SHA256")]:
    t = safe(tbl)
    if t is not None and ioc_sha256 and hashcol in t.columns:
        r = (t.filter(F.col(hashcol).isin(ioc_sha256))
               .select(F.lit("sha256").alias("type"), F.col(hashcol).alias("ioc"),
                       F.lit(tbl).alias("table"), F.col("DeviceName").alias("device"),
                       F.col("TimeGenerated").alias("time")))
        hits.append(r)

# --- Network IOCs (IP + domain) across DeviceNetworkEvents ---
net = safe("DeviceNetworkEvents")
if net is not None and ioc_ips and "RemoteIP" in net.columns:
    hits.append(net.filter(F.col("RemoteIP").isin(ioc_ips))
                   .select(F.lit("ip").alias("type"), F.col("RemoteIP").alias("ioc"),
                           F.lit("DeviceNetworkEvents").alias("table"),
                           F.col("DeviceName").alias("device"), F.col("TimeGenerated").alias("time")))
if net is not None and ioc_domains and "RemoteUrl" in net.columns:
    hits.append(net.filter(F.col("RemoteUrl").isin(ioc_domains))
                   .select(F.lit("domain").alias("type"), F.col("RemoteUrl").alias("ioc"),
                           F.lit("DeviceNetworkEvents").alias("table"),
                           F.col("DeviceName").alias("device"), F.col("TimeGenerated").alias("time")))

SUMMARY_COLS = ["type","ioc","first_seen","last_seen","hits","devices"]
if hits:
    all_hits = hits[0]
    for h in hits[1:]:
        all_hits = all_hits.unionByName(h)
    summary = (all_hits.groupBy("type","ioc")
        .agg(F.min("time").alias("first_seen"), F.max("time").alias("last_seen"),
             F.count("*").alias("hits"), F.countDistinct("device").alias("devices"))
        .orderBy("first_seen")).toPandas()
else:
    # None of the endpoint tables are readable in this workspace, or no IOCs configured.
    print("No searchable tables for the configured IOC types.")
    summary = pd.DataFrame(columns=SUMMARY_COLS)

print(f"IOC matches: {len(summary)}")
summary

## 4. Draw the campaign timeline

One horizontal bar per indicator, running from first seen to last seen, coloured by
indicator type.

**What to look for:** the left-hand edge of each bar. An indicator whose bar starts eighteen
months ago is telling you the intrusion predates the threat report by eighteen months, and
that is a very different conversation from the one you thought you were having.

If nothing hits, that is a real result and worth saying out loud. An empty chart is not the
same as a broken query.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.dates as mdates

if not summary.empty:
    summary["first_seen"] = pd.to_datetime(summary["first_seen"])
    summary["last_seen"]  = pd.to_datetime(summary["last_seen"])
    cmap = {"sha256":"#8e44ad","ip":"#c0392b","domain":"#e67e22"}
    plt.figure(figsize=(13, max(3, .5*len(summary))))
    for i, r in summary.reset_index(drop=True).iterrows():
        plt.hlines(i, r["first_seen"], r["last_seen"], color=cmap.get(r["type"],"#333"), lw=6)
        plt.plot([r["first_seen"], r["last_seen"]], [i, i], "o", color=cmap.get(r["type"],"#333"))
    plt.yticks(range(len(summary)), [f'{r.type}:{str(r.ioc)[:24]}' for r in summary.itertuples()])
    plt.title("IOC activity across full lake history (first-seen -> last-seen)")
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    plt.tight_layout(); plt.show()
else:
    print("No IOC matches in history - good news, or widen your IOC list.")

## Why this is a notebook hunt, not a KQL query

Retro-hunting a full IOC set across **years** of telemetry is the lake's signature move. The analytics tier is cost-bound to a short window; here you point one notebook at all history, get first/last-seen per indicator, and visualise the campaign timeline.